# VTC Participant-Level Descriptive Analysis

This notebook performs non-recursive, top-level CSV descriptive analysis for one file per participant.

## Run order
1. Run Cell 2 first (data ingestion).
2. Run Cell 3 next (descriptive stats + file outputs).

## Rules enforced
- Only reads `.csv` files directly in the target directory
- Does not read subfolders
- Skips bad files with warnings
- Handles missing values safely
- Saves outputs to current notebook working directory

In [ ]:
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd

# Target folder (non-recursive scan only)
input_dir = r"\\hypatia.uio.no\lh-hf-iln-sociocognitivelab\Research\HindiNet\Classification_outputs\VTC"

# Optional debug cap: set to an int (e.g., 50) for quicker test runs, or None for all files
max_files = None

# Outputs in notebook current working directory
cwd = os.getcwd()
participant_summary_path = os.path.join(cwd, "participant_summary.csv")
descriptive_stats_path = os.path.join(cwd, "descriptive_stats.txt")

required_cols = {"uid", "start_time_s", "duration_s", "label"}
label_map = {
    "FEM": "female_dur_sec",
    "MAL": "male_dur_sec",
    "KCHI": "key_child_dur_sec",
    "OCH": "other_child_dur_sec",
}

print(f"Scanning top-level CSVs in: {input_dir}", flush=True)
input_path = Path(input_dir)

try:
    if not input_path.exists():
        raise FileNotFoundError(
            f"Input directory not found or not reachable: {input_dir}"
        )
    if not input_path.is_dir():
        raise NotADirectoryError(f"Input path is not a directory: {input_dir}")

    # Non-recursive top-level scan only.
    csv_files = sorted(
        [p.name for p in input_path.iterdir() if p.is_file() and p.suffix.lower() == ".csv"]
    )
except Exception as exc:
    raise RuntimeError(
        "Failed to scan CSV input directory. Check network access/path permissions. "
        f"Original error: {exc}"
    ) from exc

if max_files is not None:
    csv_files = csv_files[:max_files]

total_files = len(csv_files)
print(f"Found {total_files} top-level CSV file(s).", flush=True)
if total_files > 0:
    preview_count = min(10, total_files)
    print(f"Showing first {preview_count} file(s):", flush=True)
    for f in csv_files[:preview_count]:
        print(" -", f, flush=True)

results = []
skipped_files = []
start_time = time.time()

for idx, filename in enumerate(csv_files, start=1):
    file_path = input_path / filename
    participant_id = os.path.splitext(filename)[0]

    # Progress heartbeat so long runs show visible status.
    elapsed = time.time() - start_time
    print(
        f"[{idx}/{total_files}] Reading {filename} | processed={len(results)} skipped={len(skipped_files)} elapsed={elapsed:.1f}s",
        flush=True,
    )

    try:
        df = pd.read_csv(file_path)
    except Exception as exc:
        print(f"WARNING: Failed to read {filename}. Skipping. Reason: {exc}", flush=True)
        skipped_files.append((filename, str(exc)))
        continue

    missing = required_cols.difference(df.columns)
    if missing:
        msg = f"Missing required columns: {sorted(missing)}"
        print(f"WARNING: {filename} skipped. {msg}", flush=True)
        skipped_files.append((filename, msg))
        continue

    # Safe handling of missing and invalid values
    df = df.copy()
    df["duration_s"] = pd.to_numeric(df["duration_s"], errors="coerce").fillna(0.0)
    df["label"] = df["label"].astype("string").str.strip().str.upper().fillna("")

    # Initialize label-specific duration sums in seconds
    label_sums = {
        "female_dur_sec": 0.0,
        "male_dur_sec": 0.0,
        "key_child_dur_sec": 0.0,
        "other_child_dur_sec": 0.0,
    }

    for label, out_name in label_map.items():
        label_sums[out_name] = df.loc[df["label"] == label, "duration_s"].sum()

    female_dur_hr = label_sums["female_dur_sec"] / 3600.0
    male_dur_hr = label_sums["male_dur_sec"] / 3600.0
    key_child_dur_hr = label_sums["key_child_dur_sec"] / 3600.0
    other_child_dur_hr = label_sums["other_child_dur_sec"] / 3600.0

    total_input_dur_hr = (
        female_dur_hr + male_dur_hr + key_child_dur_hr + other_child_dur_hr
    )

    results.append(
        {
            "participant_id": participant_id,
            "female_dur_hr": female_dur_hr,
            "male_dur_hr": male_dur_hr,
            "key_child_dur_hr": key_child_dur_hr,
            "other_child_dur_hr": other_child_dur_hr,
            "total_input_dur_hr": total_input_dur_hr,
        }
    )

participant_df = pd.DataFrame(
    results,
    columns=[
        "participant_id",
        "female_dur_hr",
        "male_dur_hr",
        "key_child_dur_hr",
        "other_child_dur_hr",
        "total_input_dur_hr",
    ],
)

total_elapsed = time.time() - start_time
print(f"\nProcessed participants: {len(participant_df)}", flush=True)
print(f"Skipped files: {len(skipped_files)}", flush=True)
print(f"Total elapsed: {total_elapsed:.1f}s", flush=True)

In [ ]:
# Descriptive analysis across participants
if participant_df.empty:
    total_mean = np.nan
    total_median = np.nan
    total_sd = np.nan

    participant_df["female_ratio"] = np.nan
    participant_df["male_ratio"] = np.nan
    participant_df["key_child_ratio"] = np.nan
    participant_df["other_child_ratio"] = np.nan
else:
    total_mean = participant_df["total_input_dur_hr"].mean()
    total_median = participant_df["total_input_dur_hr"].median()
    total_sd = participant_df["total_input_dur_hr"].std()

    denom = participant_df["total_input_dur_hr"].to_numpy()
    participant_df["female_ratio"] = np.where(
        denom > 0, participant_df["female_dur_hr"] / denom, np.nan
    )
    participant_df["male_ratio"] = np.where(
        denom > 0, participant_df["male_dur_hr"] / denom, np.nan
    )
    participant_df["key_child_ratio"] = np.where(
        denom > 0, participant_df["key_child_dur_hr"] / denom, np.nan
    )
    participant_df["other_child_ratio"] = np.where(
        denom > 0, participant_df["other_child_dur_hr"] / denom, np.nan
    )

ratio_mean = {
    "female": participant_df["female_ratio"].mean(skipna=True),
    "male": participant_df["male_ratio"].mean(skipna=True),
    "key_child": participant_df["key_child_ratio"].mean(skipna=True),
    "other_child": participant_df["other_child_ratio"].mean(skipna=True),
}

ratio_median = {
    "female": participant_df["female_ratio"].median(skipna=True),
    "male": participant_df["male_ratio"].median(skipna=True),
    "key_child": participant_df["key_child_ratio"].median(skipna=True),
    "other_child": participant_df["other_child_ratio"].median(skipna=True),
}

# Required console output
print("\n=== TOTAL INPUT (hours per hour) ===")
print(f"Mean:   {total_mean:.6f}")
print(f"Median: {total_median:.6f}")
print(f"SD:     {total_sd:.6f}")

print("\n=== INPUT COMPOSITION (MEAN %) ===")
print(f"Female:    {ratio_mean['female'] * 100:.2f}%")
print(f"Male:      {ratio_mean['male'] * 100:.2f}%")
print(f"Key Child: {ratio_mean['key_child'] * 100:.2f}%")
print(f"Other Child: {ratio_mean['other_child'] * 100:.2f}%")

# Save participant-level output
participant_df[
    [
        "participant_id",
        "female_dur_hr",
        "male_dur_hr",
        "key_child_dur_hr",
        "other_child_dur_hr",
        "total_input_dur_hr",
    ]
].to_csv(participant_summary_path, index=False)

# Save descriptive stats text output
with open(descriptive_stats_path, "w", encoding="utf-8") as f:
    f.write("=== TOTAL INPUT (hours per hour) ===\n")
    f.write(f"Mean: {total_mean:.6f}\n")
    f.write(f"Median: {total_median:.6f}\n")
    f.write(f"SD: {total_sd:.6f}\n\n")

    f.write("=== INPUT COMPOSITION (MEAN %) ===\n")
    f.write(f"Female: {ratio_mean['female'] * 100:.2f}%\n")
    f.write(f"Male: {ratio_mean['male'] * 100:.2f}%\n")
    f.write(f"Key Child: {ratio_mean['key_child'] * 100:.2f}%\n")
    f.write(f"Other Child: {ratio_mean['other_child'] * 100:.2f}%\n\n")

    f.write("=== INPUT COMPOSITION (MEDIAN %) ===\n")
    f.write(f"Female: {ratio_median['female'] * 100:.2f}%\n")
    f.write(f"Male: {ratio_median['male'] * 100:.2f}%\n")
    f.write(f"Key Child: {ratio_median['key_child'] * 100:.2f}%\n")
    f.write(f"Other Child: {ratio_median['other_child'] * 100:.2f}%\n\n")

    f.write(f"Processed participants: {len(participant_df)}\n")
    f.write(f"Skipped files: {len(skipped_files)}\n")

print("\nSaved outputs:")
print(" -", participant_summary_path)
print(" -", descriptive_stats_path)

if skipped_files:
    print("\nSkipped file details:")
    for name, reason in skipped_files:
        print(f" - {name}: {reason}")